# STEP 6.5 · 위기집단 진단 (2x2 분석틀)

### 군집분석 대신 2x2 분석틀
클러스터링은 실루엣 0.16~0.20으로 군집이 안 갈라져 부적합.
대신 명확한 기준의 2x2 분석틀을 쓴다.

### ★ 분할 기준 확정
- 가로축: **AI 활용 여부** (q38: 1=활용)
- 세로축: **디지털 피로 4점 이상** (5점 척도에서 '대체로/매우 그렇다')

피로 기준을 '4점 이상'으로 확정한 이유: 4점='대체로 그렇다',
5점='매우 그렇다'로, 둘 다 피로를 명확히 호소하는 응답이다.
중앙값(3점) 기준보다 해석이 명확하다.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT))

DATA_RAW = ROOT / "data" / "raw"
DATA_PROCESSED = ROOT / "data" / "processed"
OUT_TABLES = ROOT / "outputs" / "tables"
OUT_FIGURES = ROOT / "outputs" / "figures"

DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
OUT_TABLES.mkdir(parents=True, exist_ok=True)
OUT_FIGURES.mkdir(parents=True, exist_ok=True)

import pandas as pd, numpy as np
import warnings; warnings.filterwarnings('ignore')
dd = pd.read_csv(DATA_PROCESSED / "jtsi_2025.csv")

In [2]:
dd['hi_fatigue'] = dd['fatigue'] >= 4   # 4점 이상 = 피로 호소

names = {
    (True,  True ): '② AI 과부하형',   # 쓰는데 지침 ★타겟
    (True,  False): '① 안정 적응형',
    (False, True ): '④ 전환 소외형',   # 안 쓰는데 지침 ★타겟
    (False, False): '③ 전환 관망형',
}
total = len(dd)
rows = []
for (u,f), label in names.items():
    sub = dd[(dd['ai_user']==u) & (dd['hi_fatigue']==f)]
    rows.append([label, len(sub), round(len(sub)/total*100,1),
                 round(sub['JTSI'].mean(),2), round(sub['burnout'].mean(),2),
                 round(sub['satis'].mean(),2)])
matrix = pd.DataFrame(rows, columns=['유형','인원수','비율(%)','JTSI','번아웃','만족도'])
matrix.to_csv(OUT_TABLES / "diagnosis_2x2.csv", index=False, encoding='utf-8-sig')
matrix

,유형,인원수,비율(%),JTSI,번아웃,만족도
0,② AI 과부하형,549,27.2,1.33,3.51,5.75
1,① 안정 적응형,624,30.9,-1.11,3.06,6.35
2,④ 전환 소외형,370,18.3,1.26,3.46,5.58
3,③ 전환 관망형,477,23.6,-1.05,2.98,6.41


In [3]:
hi = matrix[matrix['유형'].str.contains('과부하|소외')]
print(f"고위험 2집단: {hi['인원수'].sum()}명 = {hi['비율(%)'].sum():.1f}%")

고위험 2집단: 919명 = 45.5%


### 진단 결과

| 사분면 | 인원 | 비율 | JTSI | 정책 의미 |
|---|---|---|---|---|
| ② AI 과부하형 | 549명 | 27.2% | +1.33 | AI 쓰지만 지침 ★타겟 |
| ④ 전환 소외형 | 370명 | 18.3% | +1.26 | AI 비활용 상태에서도 높은 피로 ★타겟 |
| ① 안정 적응형 | 624명 | 30.9% | -1.11 | 이상적 |
| ③ 전환 관망형 | 477명 | 23.6% | -1.05 | 전환 스트레스가 상대적으로 낮은 관망 집단 |

**고위험 두 집단(②④)이 919명, 약 45.5%.** ②는 'AI를 쓰지만
일이 안 줄어 지친' 사람, ④는 'AI 활용은 낮으나 디지털 피로가 높은' 집단.

> 정책은 다르게: ② 업무부담 경감+AI효율화 / ④ 기초역량+진입장벽완화

**수정 노트**: ③ '무관심형'→'전환 관망형',
비율(%) 컬럼 추가, 피로 기준을 '4점 이상'으로 명시.

